<a href="https://colab.research.google.com/github/bigbirdjones/laguardia_data_analytics/blob/main/NYC_Cooling_Tower_System_Inspection_Results_EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: LOAD THE DATA

## Main Dataset

I'll load my main dataset via API in 3 segments, then concatinate to create a full DataFrame.

My Dataset are results from inspections conducted by the Department of Health & Mental Hygiene (DOHMH) on cooling towers for large HVAC installations around the city.

These violations are covered by Chapter 8 of Title 24 of the Rules of The City of New York. (24 RCNY)

In [4]:
import pandas as pd

In [5]:
URL = "https://data.cityofnewyork.us/resource/f9wb-g8mb.csv?$limit=50000"

df1 = pd.read_csv(URL)

/tmp/ipykernel_34368/2484009375.py:3: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df1 = pd.read_csv(URL)


In [6]:
URL = "https://data.cityofnewyork.us/resource/f9wb-g8mb.csv?$limit=50000&$offset=50000"

df2 = pd.read_csv(URL)

In [8]:
URL = "https://data.cityofnewyork.us/resource/f9wb-g8mb.csv?$limit=22000&$offset=100000"

df3 = pd.read_csv(URL)

KeyboardInterrupt: 

In [ ]:
full_df = pd.concat([df1, df2, df3])
full_df

## Auxillary Dataset

Next i'll load my auxillary dataset, which after cleaning I will merge with the main dataset.

These records are derived from the Office of Administrative Trials & Hearings.

They have +20 Million records spanning over a decade, so I queried the records on NYC Open Data to get only the ones from the issuing agency i'm looking for, which is 'COOLING TOWERS - DOHMH', and downloaded the results in a csv.




In [ ]:
URL2 = "/content/drive/MyDrive/NYC Cooling Tower EDA Capstone/OATH_Hearings_Division_Case_Status_20260409.csv"
oath_df = pd.read_csv(URL2)

I'm dropping these columns immediately because they are obviously empty from even a cursory glance at the spreadsheet.

They are for charges 2 thru 10, but in the cases i'm looking at, the DOHMH seems to have only issued a single charge per ticketed violation, so it's one charge per row.

OATH handles administrative hearings for multiple city agencies, so it makes sense to leave space, but goodbye empty columns!

In [ ]:
oath_df = oath_df.drop(oath_df.columns[[42,43,44,45,46,47,48,49,50,
                                        51,52,53,54,55,56,57,58,59,60,
                                        61,62,63,64,65,66,67,68,69,70,
                                        71,72,73,74,75,76,77]], axis=1)

(I know this is technically cleaning, but it's my notebook and I'll decide where the title breaks go... speaking of which)

In [ ]:
# (I know this is technically cleaning, but it's my notebook and I'll decide where the title breaks go... speaking of which)

#Step 2: CLEAN THE DATA

## Dropping NaN's

First, anyone who wasn't issued a violation I'm not interested in, so NaN's in the **'violation_code'** column are my first target.

In [ ]:
full_df["violation_code"].isnull().sum()

In [ ]:
full_df.dropna(subset="violation_code", inplace= True)

In [ ]:
print(121987-33782)

Next, the **'summons_number'** is how I'm linking my main data set with the **'Ticket Number'** in the auxillary dataset, so NaN's there need to be dealt with as well.

In [ ]:
full_df['summons_number'].isnull().sum()

In [ ]:
full_df.dropna(subset='summons_number', inplace= True)

In [ ]:
print(88205-33764)

Then a quick datatype recast from float64 to Int64 so the key columns will match.

In [ ]:
full_df['summons_number'] = full_df['summons_number'].astype(int)

In [ ]:
full_df

In [ ]:
oath_df

## Merge DataFrames

In [ ]:
combo_df = pd.merge(full_df, oath_df, left_on='summons_number', right_on='Ticket Number')

In [ ]:
combo_df.columns

There's a lot of redundant info here, so lets make something a little cleaner.

In [ ]:
violations_df = combo_df[['Ticket Number', 'system_id', 'address', 'borough', 'zip_code', 'status', 'inspection_date','inspection_type', 'bbl', 'latitude', 'longitude', 'Violation Date', 'violation_code', 'Charge #1: Code Description', 'Respondent Last Name', 'Penalty Imposed', 'Paid Amount', 'Additional Penalties or Late Fees', 'Compliance Status']]

In [ ]:
violations_df

# Step 3: ANALYZE THE DATA

In [ ]:
import seaborn as sns